In [3]:
import sys
from collections import deque
import copy
import time


# 1. Board helpers

def parse_board(source):
    
    if isinstance(source, str):          # filename
        with open(source) as fh:
            lines = [l.strip() for l in fh if l.strip()]
    else:
        lines = source                   # already a list of strings

    assert len(lines) == 9, "Board must have exactly 9 rows."
    domains = {}
    for r, line in enumerate(lines):
        assert len(line) == 9, f"Row {r} must have exactly 9 digits."
        for c, ch in enumerate(line):
            v = int(ch)
            domains[(r, c)] = {v} if v != 0 else set(range(1, 10))
    return domains


def pretty_print(domains):
    """Render a solved (or partially solved) board with box separators."""
    sep = "+-------+-------+-------+"
    lines = [sep]
    for r in range(9):
        row = "| "
        for c in range(9):
            d = domains[(r, c)]
            val = str(next(iter(d))) if len(d) == 1 else "."
            row += val + (" | " if c % 3 == 2 else " ")
        lines.append(row)
        if r % 3 == 2:
            lines.append(sep)
    return "\n".join(lines)


# 2. Constraint structure — peers
def _compute_peers(r, c):
    """
    Return the set of all cells that share a row, column, or 3×3 box
    with cell (r, c).  Every Sudoku constraint is 'all different' over
    a peer group.
    """
    peers = set()
    # Same row
    for cc in range(9):
        if cc != c:
            peers.add((r, cc))
    # Same column
    for rr in range(9):
        if rr != r:
            peers.add((rr, c))
    # Same 3×3 box
    br, bc = 3 * (r // 3), 3 * (c // 3)
    for rr in range(br, br + 3):
        for cc in range(bc, bc + 3):
            if (rr, cc) != (r, c):
                peers.add((rr, cc))
    return frozenset(peers)


# Pre-compute once — used throughout search
PEERS = {(r, c): _compute_peers(r, c)
         for r in range(9) for c in range(9)}


# 3. AC-3 (arc consistency)

def revise(domains, xi, xj):
   
    revised = False
    for v in list(domains[xi]):
       
        if all(w == v for w in domains[xj]):
            domains[xi].discard(v)
            revised = True
    return revised


def ac3(domains):
   
    # Seed queue with all directed arcs
    queue = deque(
        (xi, xj)
        for xi in PEERS
        for xj in PEERS[xi]
    )
    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False               # domain wipe-out → failure
            # xi's domain shrank; re-check all arcs (xk → xi)
            for xk in PEERS[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True

# 4. Backtracking search

# Global counters reset before each puzzle
_bt_calls    = 0
_bt_failures = 0


def select_unassigned_variable(domains):
    """
    MRV heuristic: pick the unassigned cell with the fewest remaining
    legal values.  Returns None when every cell is assigned.
    """
    unassigned = [
        (len(d), cell)
        for cell, d in domains.items()
        if len(d) > 1
    ]
    if not unassigned:
        return None
    return min(unassigned)[1]   # cell with smallest domain


def is_complete(domains):
    """True when every cell has exactly one value assigned."""
    return all(len(d) == 1 for d in domains.values())


def backtrack(domains):
    """
    Recursive backtracking search with forward checking and AC-3
    propagation.

    Returns the solved domains dict on success, or None on failure.
    """
    global _bt_calls, _bt_failures
    _bt_calls += 1

    if is_complete(domains):
        return domains                    # solution found

    cell = select_unassigned_variable(domains)
    if cell is None:
        return None                       # shouldn't happen; safeguard

    for value in sorted(domains[cell]):  # try values in ascending order

        # Deep-copy so backtracking can restore previous state
        new_domains = copy.deepcopy(domains)
        new_domains[cell] = {value}

        # ── Forward checking ────────────────────────────────────────
        # Remove `value` from every peer's domain
        fc_ok = True
        for peer in PEERS[cell]:
            new_domains[peer].discard(value)
            if len(new_domains[peer]) == 0:
                fc_ok = False
                break

        if not fc_ok:
            _bt_failures += 1
            continue                      # prune this branch

        #  AC-3 propagation
        if not ac3(new_domains):
            _bt_failures += 1
            continue                      # prune this branch

        #  Recurse 
        result = backtrack(new_domains)
        if result is not None:
            return result                 # propagate success upward

    # All values exhausted → backtrack
    _bt_failures += 1
    return None

# 5. High-level solver
def solve(label, source):
   
    global _bt_calls, _bt_failures
    _bt_calls    = 0
    _bt_failures = 0

    domains = parse_board(source)

    # Initial AC-3 pass
    if not ac3(domains):
        print(f"\n{label}: Unsolvable — initial AC-3 wiped a domain.")
        return None

    # ── Search ──────────────────────────────────────────────────
    t_start = time.perf_counter()
    result  = backtrack(domains)
    elapsed = (time.perf_counter() - t_start) * 1000   # ms

    print(f"\n{'═'*52}")
    print(f"  {label}")
    print(f"{'═'*52}")

    if result:
        print(pretty_print(result))
    else:
        print("  No solution found.")

    print(f"\n  BACKTRACK calls    : {_bt_calls}")
    print(f"  BACKTRACK failures : {_bt_failures}")
    print(f"  Time               : {elapsed:.1f} ms")
    print()
    return result


EASY = [
    "004030050",
    "609400000",
    "005100489",
    "000060930",
    "300807002",
    "026040000",
    "453009600",
    "000004705",
    "090050200",
]


MEDIUM = [
    "530070000",
    "600195000",
    "098000060",
    "800060003",
    "400803001",
    "700020006",
    "060000280",
    "000419005",
    "000080079",
]


HARD = [
    "800000000",
    "003600000",
    "070090200",
    "060005030",
    "004010700",
    "010400600",
    "009200050",
    "000080040",
    "300001090",
]


VERY_HARD = [
    "000000010",
    "040030602",
    "000500000",
    "010070500",
    "700000300",
    "003010080",
    "000900006",
    "080100040",
    "500002007",
]

# 7. Entry point

if __name__ == "__main__":
    if len(sys.argv) == 2:
        # Single file mode
        solve(sys.argv[1], sys.argv[1])
    else:
        # Solve all four built-in boards
        solve("Easy",      EASY)
        solve("Medium",    MEDIUM)
        solve("Hard",      HARD)
        solve("Very Hard", VERY_HARD)



════════════════════════════════════════════════════
  Easy
════════════════════════════════════════════════════
+-------+-------+-------+
| 7 8 4 | 9 3 2 | 1 5 6 | 
| 6 1 9 | 4 8 5 | 3 2 7 | 
| 2 3 5 | 1 7 6 | 4 8 9 | 
+-------+-------+-------+
| 5 7 8 | 2 6 1 | 9 3 4 | 
| 3 4 1 | 8 9 7 | 5 6 2 | 
| 9 2 6 | 5 4 3 | 8 7 1 | 
+-------+-------+-------+
| 4 5 3 | 7 2 9 | 6 1 8 | 
| 8 6 2 | 3 1 4 | 7 9 5 | 
| 1 9 7 | 6 5 8 | 2 4 3 | 
+-------+-------+-------+

  BACKTRACK calls    : 1
  BACKTRACK failures : 0
  Time               : 0.0 ms


════════════════════════════════════════════════════
  Medium
════════════════════════════════════════════════════
+-------+-------+-------+
| 5 3 4 | 6 7 8 | 9 1 2 | 
| 6 7 2 | 1 9 5 | 3 4 8 | 
| 1 9 8 | 3 4 2 | 5 6 7 | 
+-------+-------+-------+
| 8 5 9 | 7 6 1 | 4 2 3 | 
| 4 2 6 | 8 5 3 | 7 9 1 | 
| 7 1 3 | 9 2 4 | 8 5 6 | 
+-------+-------+-------+
| 9 6 1 | 5 3 7 | 2 8 4 | 
| 2 8 7 | 4 1 9 | 6 3 5 | 
| 3 4 5 | 2 8 6 | 1 7 9 | 
+-------+-------+---